# 5차시 (1) BERT 한국어 감성 분류 — 파인튜닝으로 '못 하던 걸' 하게 만들기

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

4차시에서는 scikit-learn으로 **분류**(붓꽃)를 직접 만들었죠. 오늘은 같은 분류를 **딥러닝 모델(BERT)** 로, **한국어 문장** 으로 풉니다. 오늘의 흐름:

1. **약한 출발점** — 영어 감성 모델은 한국어를 못 읽고, 한국어 모델(`klue/bert-base`)도 아직 '감성 분류'를 안 배웠다(둘 다 찍기 수준).
2. **다 같이 라벨링** — 공용 Google Sheet에 한국어 문장 + 정답(긍정/부정)을 함께 모은다.
3. **파인튜닝** — 그 데이터로 `klue/bert-base` 를 학습시켜, **같은 모델의 학습 전 vs 학습 후** 를 비교한다.
4. **해석** — 모델이 내는 건 사실 **숫자(확률→번호)**, 우리가 그걸 라벨로 해석한다.

> 핵심: **모델을 바꾼 게 아니라**, *같은 모델* 에게 우리가 라벨링한 데이터로 이 과제를 **가르쳐서** 할 수 있게 만든다.

## ⚠️ 시작 전 필수: GPU 런타임 켜기

모델 학습은 **GPU** 가 있어야 빠릅니다.

1. 상단 메뉴 **[런타임] → [런타임 유형 변경]**
2. **하드웨어 가속기** 를 **T4 GPU** → **저장**
3. 다시 연결되면 아래 셀로 확인합니다. (GPU 없어도 되지만 몇 배 느립니다.)

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU 사용 가능:", torch.cuda.get_device_name(0))
else:
    print("GPU가 꺼져 있습니다. [런타임]>[런타임 유형 변경]>T4 GPU 로 설정 후 다시 실행하세요.")

### 라이브러리 설치
`transformers`(모델)·`datasets`(데이터)·`evaluate`(정확도). Colab엔 `torch`·`pandas`가 이미 있습니다.

In [ ]:
!pip install -qU transformers datasets evaluate
print("설치 완료!")

---
# Part 0 · 약한 출발점 — 영어 모델은 한국어를 모른다

먼저 **고정 테스트셋**(아래 24문장, 긍정 12 / 부정 12)을 정해 둡니다. 이 테스트셋으로 **학습 전** 과 **학습 후** 를 똑같이 채점해 비교할 거예요.

> **왜 테스트셋을 따로 두나요?** 시험 문제를 미리 알려주고 시험을 보면 실력을 알 수 없죠.
> 모델도 **학습에 쓴 문장으로 채점하면 '외운 것'** 이라 진짜 실력이 아닙니다.
> 그래서 이 24문장은 **학습에는 절대 쓰지 않고**, 학습 전/후 채점에만 씁니다 — 공정한 잣대(train/test 분리).

In [ ]:
# 고정 테스트셋 — 학습에 쓰지 않고, before/after 비교에만 사용
KO_TEST = [
    ("정말 인생 영화였어요, 강력 추천합니다", "긍정"),
    ("배송이 생각보다 훨씬 빨라서 만족스러워요", "긍정"),
    ("음식이 따뜻하고 정성이 느껴졌습니다", "긍정"),
    ("화면이 선명하고 디자인도 고급스러워요", "긍정"),
    ("직원분이 친절해서 기분 좋게 다녀왔어요", "긍정"),
    ("가격 대비 품질이 훌륭합니다", "긍정"),
    ("오랜만에 시간 가는 줄 모르고 봤네요", "긍정"),
    ("포장이 꼼꼼해서 안심이 됐어요", "긍정"),
    ("향이 은은하고 오래가서 좋아요", "긍정"),
    ("설명이 자세해서 따라 하기 쉬웠습니다", "긍정"),
    ("기대 이상으로 깔끔하게 잘 작동해요", "긍정"),
    ("다음에도 꼭 다시 이용할 생각이에요", "긍정"),
    ("기대했는데 너무 실망스러웠어요", "부정"),
    ("배송이 일주일 넘게 걸려서 짜증났습니다", "부정"),
    ("음식이 식어서 오고 맛도 별로였어요", "부정"),
    ("화면에 자꾸 줄이 가고 불량 같아요", "부정"),
    ("직원이 불친절해서 다시는 안 갈 거예요", "부정"),
    ("가격만 비싸고 품질은 형편없네요", "부정"),
    ("중간부터 너무 지루해서 졸았습니다", "부정"),
    ("포장이 엉망이라 제품이 찌그러져 왔어요", "부정"),
    ("냄새가 이상하고 금방 사라져요", "부정"),
    ("설명서가 부실해서 한참 헤맸어요", "부정"),
    ("며칠 만에 고장 나서 환불했습니다", "부정"),
    ("두 번 다시 사고 싶지 않은 제품이에요", "부정")
]

# 영어 모델이 영어엔 잘 맞는지 확인할 소량 영어 예시
EN_DEMO = [
    ("This movie was absolutely fantastic!", "positive"),
    ("Great quality and super fast shipping.", "positive"),
    ("I love this product, highly recommend it.", "positive"),
    ("This was a complete waste of time.", "negative"),
    ("Terrible quality, very disappointed.", "negative"),
    ("I would never buy this again.", "negative")
]

print("한국어 테스트셋:", len(KO_TEST), "문장 / 영어 예시:", len(EN_DEMO))

먼저 **영어 감성 모델**을 불러와 **영어 문장**에 써 봅니다. 영어엔 잘 맞을 거예요.

In [ ]:
from transformers import pipeline

en_clf = pipeline("text-classification",
                  model="distilbert-base-uncased-finetuned-sst-2-english",
                  device=0 if torch.cuda.is_available() else -1)

for text, gold in EN_DEMO:
    p = en_clf(text)[0]
    print(f"[{gold:8s}] {text}  ->  {p['label']} ({p['score']:.2f})")

이번엔 **같은 영어 모델에 한국어**를 넣어 봅니다. (POSITIVE→긍정, NEGATIVE→부정 으로 바꿔 채점)

In [ ]:
def en_predict_ko(text):
    label = en_clf(text)[0]["label"]          # POSITIVE / NEGATIVE
    return "긍정" if label == "POSITIVE" else "부정"

correct = 0
wrong_samples = []
for text, gold in KO_TEST:
    pred = en_predict_ko(text)
    if pred == gold:
        correct += 1
    else:
        wrong_samples.append((text, gold, pred))

baseline_acc = correct / len(KO_TEST)
print(f"영어 모델의 한국어 정확도(학습 전): {baseline_acc:.0%}  ({correct}/{len(KO_TEST)})")
print("\n틀린 예시 몇 개:")
for text, gold, pred in wrong_samples[:5]:
    print(f"  정답 {gold} / 예측 {pred}  <-  {text}")

보통 **50% 안팎(찍기 수준)** 이 나옵니다. 영어 모델은 한국어 문장의 의미를 모르니까요.
→ 이제 **한국어 데이터를 우리가 모아서**, 한국어 BERT를 가르쳐 봅시다.

> **잠깐 — 그럼 이 영어 모델을 한국어로 파인튜닝하면 되지 않나요?**
>
> 안 됩니다. 영어 모델의 **토크나이저** 는 한국어를 **자모(ㅇ, ㅣ, ㅎ …)로 잘게 쪼개** 의미를 못 담습니다. 그래서 한국어로 파인튜닝해도 잘 못 배웁니다.
> *(실제 실험: 영어 모델은 파인튜닝 후 오히려 **47% → 37%** 로 떨어졌고, 한국어 모델 `klue/bert-base` 는 **56% → 74%** 로 올랐습니다.)*
>
> 그래서 우리는 **한국어를 이미 이해하는 `klue/bert-base`** 를 데려와, '감성 분류' 라는 우리 과제만 우리 데이터로 가르칩니다.

### 잠깐 — 토크나이저(tokenizer)가 뭐죠? 왜 한국어/영어 모델이 다른가요?

모델은 글자를 바로 못 읽습니다. 문장을 **토큰(token)** 이라는 조각으로 자르고 **숫자로** 바꿔 주는 도구가 **토크나이저** 입니다. 모델마다 **짝** 인 토크나이저가 정해져 있어요.

- **영어 모델의 토크나이저**: 영어 조각에 맞춰져 있어, 한국어를 넣으면 **자모(ㅇ, ㅣ …)로 잘게 부수거나 `[UNK]`(모르는 토큰)** 로 처리 → 의미가 깨집니다. (그래서 영어 모델은 한국어를 못 하죠.)
- **한국어 모델(`klue/bert-base`)의 토크나이저**: 한국어 조각(형태소·자주 쓰는 단어)에 맞춰져 있어 의미를 잘 담습니다.
- **다국어 모델**(예: `bert-base-multilingual-cased`): 100여 개 언어를 한 토크나이저로 다루도록 함께 학습 — 한국어도 어느 정도 처리하지만, 특정 언어 전용 모델보다는 대개 약합니다.

아래에서 **같은 한국어 문장** 이 두 토크나이저에서 얼마나 다르게 쪼개지는지 **직접** 봅시다.

> 더 보기(수업 중 화면으로): **Hugging Face 토크나이저 강의** https://huggingface.co/learn/nlp-course/ko/chapter2/4 · **OpenAI 토크나이저 도구**(한국어 vs 영어 토큰 수 비교) https://platform.openai.com/tokenizer

In [ ]:
from transformers import AutoTokenizer

# 두 토크나이저를 준비합니다 (이 객체들로 아래에서 함수를 직접 만듭니다)
en_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")  # 영어 모델용
ko_tok = AutoTokenizer.from_pretrained("klue/bert-base")                                    # 한국어 모델용
print("토크나이저 준비 완료 — en_tok(영어), ko_tok(한국어)")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1 — 토크나이저 비교 함수 직접 만들기 (덱 &#39;토크나이저&#39; 구현)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 문장 하나를 받아 영어(<code>en_tok</code>)·한국어(<code>ko_tok</code>) 두 토크나이저로 각각 토큰화해 결과를 나란히 출력하는 <code>show_tokens(text)</code> 함수를 완성하세요.<br><b>힌트</b> &nbsp; 토크나이저 객체의 <code>.tokenize(문장)</code> 메서드가 토큰 리스트를 돌려줍니다. <code>en_tok</code>·<code>ko_tok</code> 각각에 부른 뒤, 문장·영어토큰·한국어토큰을 <code>print</code> 하세요.<br><b>예상 동작</b> &nbsp; 영어 토크나이저는 한글을 <b>자모·<code>[UNK]</code>·조각</b> 으로 잘게, 한국어 토크나이저는 <b>의미 있는 단어 조각</b> 으로 나눕니다.
</div>
</div>

In [ ]:
# ✅ 연습 1 정답 — 토크나이저 비교 함수
def show_tokens(text):
    en = en_tok.tokenize(text)
    ko = ko_tok.tokenize(text)
    print("문장:", text)
    print("  영어 토크나이저:", en)
    print("  한국어 토크나이저:", ko)

show_tokens("이 영화 정말 재미있어요")
show_tokens("오늘 점심은 정말 맛있었어요")

---
# Part A · 다 같이 데이터 라벨링하기 (Google Sheet)

좋은 분류기엔 **정답(label)이 달린 데이터** 가 필요합니다. 오늘은 **감성(긍정/부정)** 으로, 반 전체가 **Google Sheet 한 장** 을 10~20분간 함께 채웁니다. 한 사람이 몇 문장씩만 적어도 금세 **수백 개** 가 모입니다.

## A-1. 다 같이 라벨링 — **여러분이 할 일**

1. **공용 시트 링크** 를 엽니다(수업 중 공유). 이미 **약 500개** 문장이 `text`·`label` 로 채워져 있어요.
2. **맨 아래 빈 줄** 로 내려가, 여러분이 생각한 문장을 **한 줄씩** 추가합니다.
   - `text` 칸: 한국어 문장 — 예) `화면은 예쁜데 배터리가 너무 빨리 닳아요`
   - `label` 칸: `긍정` 또는 `부정` **둘 중 하나** (오타·공백 없이! `긍정 `(뒤 공백)·`positive` 섞이면 다른 클래스로 취급)
3. **애매한 문장** 을 특히 환영합니다 — 모델이 어려워하는 걸 가르치면 더 똑똑해져요.
4. 한 사람이 **3~5문장** 만 추가해도 반 전체로 금세 수백 개가 모입니다.

| text | label |
|---|---|
| 이 영화 진짜 재밌어요 | 긍정 |
| 스토리가 너무 지루했다 | 부정 |
| (여러분이 추가) | 긍정/부정 |

> 아래 셀이 이 공용 시트를 CSV로 **바로 불러옵니다**(시트ID가 이미 들어 있음).

## A-2. 시트를 CSV로 불러오기

시트 주소의 **시트ID** 만 있으면 `pandas` 가 `.../export?format=csv` 주소로 바로 읽습니다.
아래 셀에는 **우리 반 공용 시트ID가 이미 들어 있습니다.** (그대로 실행)

In [ ]:
# 우리 반 공용 라벨링 시트 (링크 공개 · 이미 500여 개 채워져 있음)
SHEET_ID = "14QwAR_rWhrLya8MJSHmjQLG17ztO9jax7vRX5zGkWuo"
GID = "0"

csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
print("CSV 주소:", csv_url)

## A-3. 데이터 불러오기 (pandas)
시트가 아직이거나 에러가 나면, **다음 셀의 예비 데이터**(60문장)로 바로 진행할 수 있습니다.

In [ ]:
import pandas as pd

try:
    df = pd.read_csv(csv_url)
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    print("시트에서 불러오기 성공! 데이터 수:", len(df))
    print(df["label"].value_counts())
    USE_SHEET = True
except Exception as e:
    print("시트를 불러오지 못했습니다 (준비 전이거나 공유 설정 확인).")
    print("오류:", e, "\n=> 아래 예비 데이터로 진행하세요.")
    USE_SHEET = False

## A-4. (예비) 시트가 없을 때 — 내장 학습 데이터 60문장
위 A-3 가 성공했다면 건너뛰어도 됩니다(시트 데이터를 덮어쓰지 않습니다).

In [ ]:
import pandas as pd

if not USE_SHEET:
    backup = [
        ("이 영화 정말 재미있었어요", "긍정"),
        ("연기가 너무 자연스럽고 좋았다", "긍정"),
        ("스토리가 탄탄해서 몰입했어요", "긍정"),
        ("음악이 분위기와 잘 어울렸다", "긍정"),
        ("기대 이상으로 만족스러웠습니다", "긍정"),
        ("다시 보고 싶은 작품이에요", "긍정"),
        ("배우들의 호흡이 환상적이었다", "긍정"),
        ("끝까지 긴장감이 넘쳤어요", "긍정"),
        ("가족과 보기 좋은 따뜻한 영화", "긍정"),
        ("여운이 오래 남는 명작이다", "긍정"),
        ("제품이 튼튼하고 마감이 깔끔해요", "긍정"),
        ("생각보다 가벼워서 들고 다니기 편해요", "긍정"),
        ("색감이 예쁘고 사진이 잘 나와요", "긍정"),
        ("가성비가 정말 뛰어난 것 같아요", "긍정"),
        ("사용법이 간단해서 바로 익혔어요", "긍정"),
        ("배터리가 오래가서 만족합니다", "긍정"),
        ("맛이 깔끔하고 자극적이지 않아요", "긍정"),
        ("양도 푸짐하고 가격도 착해요", "긍정"),
        ("매장이 깨끗하고 분위기가 좋네요", "긍정"),
        ("응대가 빠르고 정확해서 편했어요", "긍정"),
        ("택배가 다음 날 바로 도착했어요", "긍정"),
        ("선물용으로 포장이 예뻐서 좋았어요", "긍정"),
        ("기능이 다양해서 활용도가 높아요", "긍정"),
        ("소음이 거의 없어서 조용해요", "긍정"),
        ("피부에 순하고 발림성이 좋아요", "긍정"),
        ("설치가 쉬워서 금방 끝났습니다", "긍정"),
        ("화질이 또렷하고 음질도 훌륭해요", "긍정"),
        ("오래 써도 질리지 않는 디자인이에요", "긍정"),
        ("문의에 친절하게 답변해 주셨어요", "긍정"),
        ("전체적으로 아주 만족스러운 경험이었어요", "긍정"),
        ("시간이 너무 아까웠어요", "부정"),
        ("내용이 지루하고 늘어졌다", "부정"),
        ("연기가 어색해서 몰입이 안 됐다", "부정"),
        ("결말이 너무 허무했어요", "부정"),
        ("기대했는데 실망스러웠습니다", "부정"),
        ("스토리가 산만하고 엉성했다", "부정"),
        ("두 번 다시 보고 싶지 않아요", "부정"),
        ("돈이 아까운 영화였다", "부정"),
        ("전개가 뻔하고 식상했어요", "부정"),
        ("소리만 시끄럽고 알맹이가 없다", "부정"),
        ("제품이 약해서 금방 망가졌어요", "부정"),
        ("생각보다 무거워서 불편해요", "부정"),
        ("색이 사진과 완전히 달라요", "부정"),
        ("가격에 비해 품질이 너무 떨어져요", "부정"),
        ("설명이 부족해서 쓰기 어려웠어요", "부정"),
        ("배터리가 금방 닳아서 불편합니다", "부정"),
        ("맛이 너무 짜고 느끼했어요", "부정"),
        ("양이 적은데 가격은 비싸요", "부정"),
        ("매장이 지저분하고 정신없었어요", "부정"),
        ("응대가 느리고 불친절했어요", "부정"),
        ("택배가 너무 늦게 와서 답답했어요", "부정"),
        ("포장이 허술해서 깨져 있었어요", "부정"),
        ("기능이 부실해서 쓸 데가 없어요", "부정"),
        ("소음이 심해서 신경 쓰여요", "부정"),
        ("피부에 트러블이 생겨서 버렸어요", "부정"),
        ("조립이 복잡해서 한참 걸렸습니다", "부정"),
        ("화질이 흐릿하고 음질도 별로예요", "부정"),
        ("디자인이 촌스럽고 마감이 거칠어요", "부정"),
        ("문의해도 답이 없어서 답답했어요", "부정"),
        ("전체적으로 다시는 안 살 것 같아요", "부정")
    ]
    df = pd.DataFrame(backup, columns=["text", "label"])
    print("예비 데이터 사용. 데이터 수:", len(df))
    print(df["label"].value_counts())
else:
    print("시트 데이터를 사용 중이라 예비 데이터는 건너뜁니다.")

---
# Part B · 데이터 준비 (라벨→숫자, 토큰화)

모델은 글자 label(`긍정`/`부정`)을 못 읽으니 **숫자 id** 로 바꿉니다. **학습**엔 우리가 모은 데이터(`df`)를, **평가**엔 Part 0의 고정 `KO_TEST` 를 씁니다(영어 모델과 같은 잣대로 비교).

In [ ]:
label_names = sorted(df["label"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}
print("클래스:", label2id)

df["label_id"] = df["label"].map(label2id)
test_df = pd.DataFrame(KO_TEST, columns=["text", "label"])
test_df["label_id"] = test_df["label"].map(label2id)
print(f"학습 {len(df)}개 / 평가(고정 테스트셋) {len(test_df)}개")

Hugging Face `datasets` 로 바꾸고, 모델과 **짝** 인 토크나이저로 문장을 토큰(숫자)으로 변환합니다.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "klue/bert-base"   # 한국어로 사전학습된 BERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(df[["text","label_id"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text","label_id"]], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_ds = train_ds.map(tokenize, batched=True).rename_column("label_id","labels")
test_ds  = test_ds.map(tokenize, batched=True).rename_column("label_id","labels")
print("토큰화 완료:", train_ds.column_names)

---
# Part C · BERT 파인튜닝 (모델 학습)

사전학습된 BERT 위에 **우리 클래스 수에 맞는 분류층** 을 얹어 학습합니다. 처음 실행 시 모델(수백 MB)을 내려받느라 잠깐 걸립니다.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names), id2label=id2label, label2id=label2id)
print("모델 준비 완료. 클래스 수:", len(label_names))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 2 — 감성 예측 함수 predict(text) 직접 만들기 (덱 &#39;숫자→라벨 해석&#39; 구현)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 문장 하나를 받아 <code>model</code> 로 예측하고, 그 결과를 우리 라벨(<b>긍정/부정</b>) 문자열로 돌려주는 <code>predict(text)</code> 함수를 <b>처음부터</b> 작성하세요.<br><b>힌트(무엇을·어떤 도구)</b> &nbsp; ① <code>tokenizer</code> 로 문장을 텐서로 바꿔 <code>model</code> 에 넣어 <code>logits</code> 를 얻습니다(계산만 하므로 <code>torch.no_grad</code> 안에서). ② 여러 클래스 점수 중 가장 높은 <b>번호</b> 를 고르는 건 <code>argmax</code>. ③ 그 번호를 사람이 읽는 라벨로 바꾸는 규칙은 <code>id2label</code>. 이 셋을 조합해 라벨 문자열을 <code>return</code> 하세요.<br><b>예상 동작</b> &nbsp; <code>predict("이 영화 재미있어요")</code> 처럼 부르면 <code>&#39;긍정&#39;</code>/<code>&#39;부정&#39;</code> 문자열이 나옵니다. (지금은 학습 전이라 자주 틀립니다 — 학습 후 같은 함수를 그대로 씁니다.)
</div>
</div>

In [ ]:
# ✅ 연습 2 정답 — 감성 예측 함수
#    ⚠️ Colab(GPU): model / tokenizer / id2label 을 사용합니다.
import torch

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = int(logits.argmax())      # 점수가 가장 높은 '번호'
    return id2label[pred_id]            # 번호 -> 라벨 (우리가 정한 규칙)

print(predict("이 영화 정말 재미있어요"))   # 학습 전이라 맞을 수도, 틀릴 수도 있습니다

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 3 — 정확도 계산 함수 accuracy_on_testset() 직접 만들기 (덱 &#39;학습 전/후 비교&#39; 구현)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 고정 테스트셋 <code>KO_TEST</code> 의 모든 (문장, 정답) 을 <code>predict</code> 로 예측해, <b>맞은 개수 ÷ 전체</b> 정확도를 돌려주는 <code>accuracy_on_testset()</code> 함수를 <b>처음부터</b> 작성하세요. (4차시 정확도 채점과 같은 원리)<br><b>힌트</b> &nbsp; <code>KO_TEST</code> 는 <code>(문장, 정답라벨)</code> 튜플들의 리스트입니다. 반복문으로 하나씩 꺼내 <code>predict(문장)</code> 과 정답을 비교하고, 같으면 카운터를 1 늘리세요. 마지막에 <code>(맞은 수 / 전체 수)</code> 를 <code>return</code>.<br><b>예상 동작</b> &nbsp; 0.0~1.0 사이 값. 학습 전에는 <b>0.5(찍기)</b> 근처가 나옵니다.
</div>
</div>

In [ ]:
# ✅ 연습 3 정답 — 정확도 계산 함수
def accuracy_on_testset():
    correct = 0
    for text, gold in KO_TEST:
        if predict(text) == gold:
            correct += 1
    return correct / len(KO_TEST)

print(f"현재 정확도: {accuracy_on_testset():.0%}")

**학습 전 성적표.** 방금 만든 모델은 분류층이 *랜덤* 이라 아직 감성을 못 맞힙니다. **여러분이 만든 함수**로 점수를 재 두고, 학습 후와 비교합니다.

In [ ]:
# 학습 '전' 점수 — 여러분이 만든 accuracy_on_testset() 으로 측정 (분류층이 아직 랜덤)
#    ⚠️ Colab(GPU): 위 연습 2·3 을 완성한 뒤 실행합니다.
before_acc = accuracy_on_testset()
print(f"학습 전 klue/bert-base 정확도: {before_acc:.0%}")
print("→ 한국어는 '이해'하지만 '감성 분류'는 아직 안 배워 찍기 수준입니다. 이제 가르쳐 봅시다.")

**정확도** 를 지표로, `Trainer` 로 학습합니다. 데이터가 적으면 GPU에서 1~3분이면 끝납니다.

In [ ]:
import numpy as np, evaluate
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, axis=-1), references=labels)

args = TrainingArguments(
    output_dir="bert_classifier", num_train_epochs=5,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    learning_rate=2e-5, eval_strategy="epoch", logging_steps=10, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()

---
# Part D · 학습 전 vs 학습 후 — **같은 모델** 로 비교

파인튜닝의 효과를 정직하게 보려면 **모델은 그대로 두고** *학습 전* 과 *학습 후* 만 비교합니다. 방금 만든 `accuracy_on_testset()` 을 **학습 후에 그대로 다시 불러**, 같은 잣대로 점수를 잽니다.

In [ ]:
# 학습 '후' 점수 — 같은 함수를 그대로 다시 사용 (model 이 학습돼 있으니 predict 도 똑똑해짐)
after_acc = accuracy_on_testset()
print("=== 같은 모델(klue/bert-base) · 같은 테스트셋 24문장 ===")
print(f"학습 전 (파인튜닝 전)          : {before_acc:.0%}")
print(f"학습 후 (우리 데이터로 파인튜닝) : {after_acc:.0%}")
print(f"향상폭                         : {after_acc - before_acc:+.0%}")
print("\n※ 오른 정확도는 '모델 교체'가 아니라 '같은 모델을 우리 데이터로 가르친' 효과입니다.")

### 눈으로 보기 — 파인튜닝 전 vs 후
막대그래프로 보면, 우리가 데이터로 가르친 효과가 한눈에 보입니다.

In [ ]:
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib          # 한글 폰트 자동 설정 (Colab)
except Exception:
    import subprocess; subprocess.run(["pip","install","-q","koreanize-matplotlib"]); import koreanize_matplotlib

labels = ["학습 전\n(파인튜닝 전)", "학습 후\n(파인튜닝 후)"]
values = [before_acc*100, after_acc*100]
plt.figure(figsize=(5.5, 4))
bars = plt.bar(labels, values, color=["#C8483B", "#2F9E6E"], width=0.5)
plt.axhline(50, ls="--", color="gray", alpha=0.6)
plt.text(1.45, 51, "찍기 수준(50%)", color="gray", ha="right", fontsize=9)
plt.ylim(0, 100); plt.ylabel("테스트 정확도 (%)")
plt.title("같은 모델(klue/bert-base) — 파인튜닝 전 vs 후")
for b, v in zip(bars, values):
    plt.text(b.get_x()+b.get_width()/2, v+2, f"{v:.0f}%", ha="center", fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()
print(f"→ 데이터로 가르치자 정확도가 {before_acc:.0%} 에서 {after_acc:.0%} 로 올랐습니다. 이게 파인튜닝(미세조정)의 힘입니다.")

### 잠깐 — 여러분의 `predict()` 안에서 무슨 일이 일어났을까요?

여러분이 만든 `predict()` 는 사실 model 이 낸 **숫자** 를 라벨로 바꾼 것입니다. 모델은 `긍정/부정` 이라는 **글자를 모르고**, 클래스마다 **점수(확률)** 를 낸 뒤 **가장 높은 쪽 번호**(2진 분류라 `0` 또는 `1`)를 고를 뿐이죠. 그 번호를 사람이 읽는 라벨로 바꾸는 건 우리가 정의한 `id2label` 규칙입니다. 아래에서 그 **내부** 를 한 번 뜯어봅시다.

In [ ]:
import torch
text = "이건 정말 인생 영화예요"
inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits[0]        # 원점수(로짓) - 클래스마다 하나씩, 숫자 2개
probs = torch.softmax(logits, dim=-1)          # 확률로 변환 (합 = 1)
pred_id = int(probs.argmax())                  # 확률이 더 높은 쪽 '번호'(0 또는 1)

print("문장:", text)
print("클래스별 확률:", {id2label[i]: round(float(probs[i]), 3) for i in range(len(probs))})
print("모델이 고른 번호:", pred_id)
print("우리가 정의한 규칙 id2label:", id2label)
print(f"→ 번호 {pred_id} 를 우리 라벨로 해석하면: '{id2label[pred_id]}'")

> **생각해 보기**: 만약 `label2id` 를 반대로 정의했다면 결과 라벨은 어떻게 될까요? 라벨이 3개(긍정·부정·**중립**)라면 모델이 내는 점수(확률)는 몇 개가 될까요?

---
# Part E · 내가 만든 함수로, 내 문장들 예측하기

학습한 모델과 **여러분이 만든 `predict()`** 로, 직접 쓴 문장들을 한 번에 분류해 봅시다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 연습 4 — 내 문장 여러 개를 반복문으로 예측하기</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 긍정 1 · 부정 1 · 애매한 문장 1 (원하면 더) 을 리스트 <code>my_sentences</code> 에 담고, <b>반복문으로</b> 하나씩 <code>predict</code> 에 넣어 <code>[예측라벨] 문장</code> 형태로 출력하는 코드를 직접 작성하세요.<br><b>힌트</b> &nbsp; 리스트를 <code>for</code> 로 돌며 각 문장에 <code>predict(문장)</code> 을 불러 라벨을 얻고, 문장과 함께 <code>print</code>. (연습 2에서 만든 <code>predict</code> 를 그대로 활용)<br><b>예상 동작</b> &nbsp; <b>모델이 틀리는 문장</b>을 하나라도 찾으면 성공! 그런 문장을 시트에 모아 다시 학습하면 정확도가 오릅니다 — "데이터가 곧 성능".
</div>
</div>

In [ ]:
# ✅ 연습 4 정답 — 내 문장들을 반복문으로 예측
my_sentences = [
    "연출이 뛰어나고 배우들의 연기가 정말 좋았다",   # 긍정
    "기대하고 갔는데 서비스가 너무 별로였어요",      # 부정
    "화면은 예쁜데 배터리가 너무 빨리 닳아요",       # 애매 (긍정+부정 섞임)
]
for s in my_sentences:
    print(f"[{predict(s)}]  {s}")

# 💡 애매한 문장에서 모델이 헷갈리기 쉽습니다. 그런 문장을 시트에 모아 다시 학습하면 좋아집니다.

---
## 마무리

- **학습 전**: 우리 모델(`klue/bert-base`)도 '감성 분류'는 아직 안 배워 찍기 수준(≈50%)이었습니다. (영어 모델은 한국어를 아예 못 읽고요.)
- **우리가 한 일**: 한국어 문장에 긍정/부정을 **직접 라벨링** → 그 데이터로 **같은 모델을 파인튜닝**.
- **학습 후**: 같은 모델·같은 테스트셋에서 정확도가 크게 올랐습니다. → **모델을 바꾼 게 아니라, 데이터로 가르쳐서** 할 수 있게 됐다.
- 모델이 내는 예측은 결국 **숫자(확률 → 번호)**, 그 번호를 우리가 정한 라벨 체계(`id2label`)로 해석합니다.
- 데이터가 많고 라벨이 깔끔할수록 더 좋아집니다 — "데이터가 곧 성능".

> 모델 저장: `trainer.save_model("my_bert")` → 나중에 `pipeline(model="my_bert")` 로 다시 사용.